# 17 — Governance and Stock-Return *Volatility* (a separated thread, with an out-of-sample test)

The main study (`09`) tested governance → forward **returns/alpha** and found a clean null.
But across every cut, the one relationship that repeatedly flickered was governance →
lower **volatility** — TRINDEX (transparency/remuneration) and OINDEX (ownership) into the
various risk measures. That is exactly where the corporate-governance literature expects an
effect to appear *first*: better-governed firms should blow up less, not necessarily earn
more. This notebook pulls that thread on its own terms.

**What makes this more than re-slicing the same data to chase the thing that already looked
good** (which the Romano-Wolf machinery elsewhere exists precisely to forbid):

- **A temporally held-out test, plus an explicit check on its limits.** The `09` panel is
  FY23–FY24. The CG scores, filing dates, controls, and the price snapshot *also* contain
  **FY25**, which no prior analysis touched. FY23–FY24 states the hypothesis ("TRINDEX and
  OINDEX predict lower forward risk"); the family (2 indices × 4 risk measures = 8) is
  fixed *before* looking at FY25 — no post-hoc narrowing — and Romano-Wolf corrects it.
  **But a held-out year is not a held-out sample:** it is the same firms, and §6 shows both
  governance and volatility are strongly persistent at the firm level, so a stable
  cross-sectional association reproduces in FY25 partly by construction. §6 therefore adds
  the **within-firm first-difference test**, which removes that confound and is the honest
  arbiter of whether anything here is a governance *effect* rather than a firm *attribute*.
- **Volatility dodges both blockers that pinned `09` at T=2.** The README documents that
  the panel cannot extend forward because (a) FY25's 252-day *return* windows haven't fully
  elapsed and (b) `ff5mom_factors_monthly.csv` has no recoverable provenance past mid-2025.
  Neither binds here: risk is measured from the stock's own returns plus the Nifty (no
  factor file needed), and volatility is well-estimated from ~6 months of daily data — so a
  **126-trading-day** horizon gives a complete FY25 window for the whole universe *today*.
- **A sharper question via a lagged-volatility control.** Volatility is highly persistent,
  so a raw cross-section mostly compares intrinsically-calm firms to intrinsically-jumpy
  ones. Adding each firm's *pre-filing* 126-day volatility as a control turns the test into
  the conditional one that actually matters: **given how volatile a firm already was, does
  its governance predict where volatility goes next?**

Fully offline from the committed price snapshot; no live market data.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import time
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
from scipy.stats import norm

BASE = Path.cwd()
PROC = BASE / 'data' / 'processed'

RF_DAILY = 0.065 / 252
FWD_WIN, LAG_WIN, MIN_OBS = 126, 126, 90     # 126 td ~ 6 months; complete for all of FY25
FYS_ALL = ['FY23', 'FY24', 'FY25']
CTRL = ['Beta_Market', 'Momentum', 'Log_MarketCap', 'DE_Ratio']

# Pre-declared confirmatory family (fixed from the FY23-24 evidence, BEFORE touching FY25):
FAMILY_IDX = ['TRINDEX', 'OINDEX']
VOLS       = ['total_vol', 'idio_vol', 'downside_vol', 'max_drawdown']
# lagged-vol control matched to each outcome (idio/downside use residual-vol lag proxy = idio)
LAGMAP = {'total_vol': 'lag_total_vol', 'idio_vol': 'lag_idio_vol',
          'downside_vol': 'lag_idio_vol', 'max_drawdown': 'lag_total_vol'}
RW_B = 2000
SEED = 42

imap = pd.read_excel(PROC / 'industry_map.xlsx')[['BSE Code', 'NSE Symbol', 'Industry', 'Sector']].dropna(subset=['NSE Symbol'])
imap['BSE Code'] = pd.to_numeric(imap['BSE Code'], errors='coerce')

snap = pd.read_parquet(PROC / 'prices_daily.parquet')
nifty = np.log(snap['^CRSLDX'] / snap['^CRSLDX'].shift(1)).dropna()
px = snap.reindex(columns=[f'{s}.NS' for s in imap['NSE Symbol']])
valid = px.columns[px.isna().mean() < 0.5].tolist()
ret_d = np.log(px[valid] / px[valid].shift(1)).dropna(how='all')
tdays = ret_d.index

fdb = pd.read_csv(PROC / 'filing_dates_db.csv')
fdb['BSE_Code'] = pd.to_numeric(fdb['BSE_Code'], errors='coerce')
fdb['Filing_Date'] = pd.to_datetime(fdb['Filing_Date'])
print(f'Snapshot: {ret_d.index.min().date()} -> {ret_d.index.max().date()}, {len(valid)} tickers')

Snapshot: 2019-07-02 -> 2026-07-22, 239 tickers


## 1 — Risk-measure construction

For each firm-FY: observe the last CG filing of the year, then measure a **forward** 126-td
window (starting +2 td, avoiding the filing-day microstructure) and a **backward** 126-td
window ending the day before the filing (the lagged control). Four risk measures per window:

- `total_vol` — SD of daily excess returns
- `idio_vol` — SD of CAPM residuals (stock vs Nifty 500; **no factor file needed**)
- `downside_vol` — SD of the negative CAPM residuals only
- `max_drawdown` — worst peak-to-trough decline over the window (a genuine tail-risk measure,
  new here — day-to-day SD misses the blow-ups governance theory is really about)

In [2]:
def vol_block(ticker, window):
    stk = ret_d.loc[window, ticker]
    a = pd.concat([stk.rename('s'), nifty.reindex(window).rename('m')], axis=1).dropna()
    if len(a) < MIN_OBS:
        return None
    s_ex, m_ex = a['s'] - RF_DAILY, a['m'] - RF_DAILY
    capm = sm.OLS(s_ex.values, sm.add_constant(m_ex.values)).fit()
    r = capm.resid
    cum = np.exp(a['s'].cumsum())
    return {'total_vol': float(s_ex.std(ddof=1)),
            'idio_vol': float(r.std(ddof=1)),
            'downside_vol': float(r[r < 0].std(ddof=1)) if (r < 0).any() else np.nan,
            'max_drawdown': float(-(cum / cum.cummax() - 1).min())}

def obs_dates(fy):
    qs = [f'Q{q}{fy}' for q in range(1, 5)]
    o = fdb[fdb['Q_FY'].isin(qs)].groupby('BSE_Code')['Filing_Date'].max().reset_index()
    return o.merge(imap, left_on='BSE_Code', right_on='BSE Code', how='left').dropna(subset=['NSE Symbol'])

rows = []
for fy in FYS_ALL:
    n_fwd = 0
    for _, rr in obs_dates(fy).iterrows():
        tk = f"{rr['NSE Symbol']}.NS"
        if tk not in ret_d.columns:
            continue
        p0 = tdays.searchsorted(rr['Filing_Date'])
        fpos = p0 + 2
        fwd = vol_block(tk, tdays[fpos:fpos + FWD_WIN]) if fpos + FWD_WIN <= len(tdays) else None
        if fwd is None:
            continue
        n_fwd += 1
        row = {'BSE Code': rr['BSE Code'], 'FY': fy, 'Industry': rr['Industry'], 'Sector': rr['Sector'], **fwd}
        lag = vol_block(tk, tdays[max(p0 - LAG_WIN, 0):p0]) if p0 - LAG_WIN >= 0 else None
        if lag is not None:
            row['lag_total_vol'], row['lag_idio_vol'] = lag['total_vol'], lag['idio_vol']
        rows.append(row)
    print(f'{fy}: {n_fwd} complete forward-vol windows')
vol = pd.DataFrame(rows)
print(f'Total firm-FY risk rows: {len(vol)}  (horizon = {FWD_WIN} td)')

FY23: 239 complete forward-vol windows


FY24: 239 complete forward-vol windows


FY25: 239 complete forward-vol windows
Total firm-FY risk rows: 717  (horizon = 126 td)


## 2 — Assemble the panel

Same governance scoring and controls as `09`'s primary spec: raw `Avg_Score` averaged to
firm-FY, Van der Waerden-transformed within each FY cross-section; Q4 controls; singleton
industries dropped so the CG coefficient is identified off genuine within-industry
variation.

In [3]:
def vdw(s):
    n = s.notna().sum()
    if n < 2:
        return pd.Series(np.nan, index=s.index)
    r = s.rank(method='average', na_option='keep')
    return r.map(lambda x: norm.ppf(x / (n + 1)) if pd.notna(x) else np.nan)

def winsor(s, lo=.01, hi=.99):
    q = s.quantile([lo, hi]); return s.clip(q.iloc[0], q.iloc[1])

def drop_singletons(df):
    c = df.groupby('Industry')['BSE Code'].nunique()
    sing = df.loc[df['Industry'].isin(c[c == 1].index), 'BSE Code'].unique()
    return df[~df['BSE Code'].isin(sing)].copy()

scores = pd.read_csv(PROC / 'cg_scores.csv')
scores['BSE Code'] = pd.to_numeric(scores['BSE Code'], errors='coerce')
scores['FY'] = 'FY' + scores['Q_FY'].str[-2:]
cg = scores[scores['FY'].isin(FYS_ALL)]
cg_avg = cg.groupby(['BSE Code', 'FY', 'Category'])['Avg_Score'].mean().reset_index()
cg_avg['vdw'] = cg_avg.groupby(['FY', 'Category'])['Avg_Score'].transform(vdw)
cg_wide = cg_avg.pivot_table(index=['BSE Code', 'FY'], columns='Category', values='vdw').reset_index()
cg_wide.columns.name = None

ctrl = pd.read_csv(PROC / 'controls_quarterly.csv')
ctrl['BSE Code'] = pd.to_numeric(ctrl['BSE Code'], errors='coerce')
ctrl_q4 = ctrl[ctrl['Q_FY'].str.startswith('Q4')].copy()
ctrl_q4['FY'] = 'FY' + ctrl_q4['Q_FY'].str[-2:]
ctrl_q4 = ctrl_q4[['BSE Code', 'FY'] + CTRL]

panel = vol.merge(cg_wide, on=['BSE Code', 'FY'], how='inner').merge(ctrl_q4, on=['BSE Code', 'FY'], how='inner')
panel_train = drop_singletons(panel[panel['FY'].isin(['FY23', 'FY24'])])
panel_test  = drop_singletons(panel[panel['FY'] == 'FY25'])
print(f'Hypothesis-gen (FY23-24): {len(panel_train)} rows, {panel_train["BSE Code"].nunique()} firms')
print(f'Held-out (FY25)         : {len(panel_test)} rows, {panel_test["BSE Code"].nunique()} firms')

Hypothesis-gen (FY23-24): 420 rows, 210 firms
Held-out (FY25)         : 210 rows, 210 firms


In [4]:
# Estimator + Romano-Wolf machinery (firm-clustered SE; same procedure as 09/10)
def reg(df, y, cg_col, lag_ctrl=None):
    d = df.copy()
    fe = pd.concat([pd.get_dummies(d['Industry'], prefix='_I', drop_first=True),
                    pd.get_dummies(d['FY'], prefix='_Y', drop_first=True)], axis=1)
    d = pd.concat([d, fe], axis=1)
    xc = [cg_col] + CTRL + ([lag_ctrl] if lag_ctrl else []) + list(fe.columns)
    sub = d[[y] + xc + ['BSE Code']].dropna()
    if sub['BSE Code'].nunique() < 10 or len(sub) < 30:
        return None
    sub = sub.copy(); sub[y] = winsor(sub[y])
    X = sm.add_constant(sub[xc].astype(float), has_constant='add')
    r = sm.OLS(sub[y].astype(float), X).fit().get_robustcov_results('cluster', groups=sub['BSE Code'].values)
    p = pd.Series(np.asarray(r.params), index=X.columns); se = pd.Series(np.asarray(r.bse), index=X.columns)
    pv = pd.Series(np.asarray(r.pvalues), index=X.columns)
    return {'beta': p[cg_col], 'se': se[cg_col], 't': p[cg_col] / se[cg_col], 'p_raw': pv[cg_col], 'N': len(sub)}

def cluster_ols_fast(y, X, cl, ncl, ti=1):
    n, k = X.shape; XtX = X.T @ X
    inv = np.linalg.pinv(XtX) if np.linalg.cond(XtX) > 1e12 else np.linalg.inv(XtX)
    b = inv @ (X.T @ y); resid = y - X @ b; Xr = X * resid[:, None]
    S = np.zeros((ncl, k)); np.add.at(S, cl, Xr)
    dof = (ncl / max(ncl - 1, 1)) * ((n - 1) / max(n - k, 1))
    vcov = inv @ (S.T @ S) @ inv * dof; sd = np.sqrt(np.maximum(np.diag(vcov), 0))
    return b[ti], (b[ti] / sd[ti] if sd[ti] > 0 else np.nan)

def build_hyp(df, pairs, lagged):
    fe = pd.concat([pd.get_dummies(df['Industry'], prefix='_I', drop_first=True),
                    pd.get_dummies(df['FY'], prefix='_Y', drop_first=True)], axis=1)
    d = pd.concat([df.reset_index(drop=True), fe.reset_index(drop=True)], axis=1)
    fe_cols = list(fe.columns); hyp = {}
    for cg_col, y in pairs:
        extra = [LAGMAP[y]] if lagged else []
        cols = [y, cg_col] + CTRL + extra + fe_cols + ['BSE Code']
        sub = d[cols].dropna().reset_index(drop=True)
        if sub['BSE Code'].nunique() < 10 or len(sub) < 30:
            continue
        yv = winsor(sub[y]).values.astype(float)
        X = np.column_stack([np.ones(len(sub)), sub[cg_col].values,
                             sub[CTRL + extra].values, sub[fe_cols].values]).astype(float)
        fr = pd.Series(np.arange(len(sub))).groupby(sub['BSE Code'].values).apply(np.array).to_dict()
        hyp[(cg_col, y)] = {'y': yv, 'X': X, 'fr': fr, 'firms': np.array(list(fr.keys()))}
    return hyp

def _codes(d):
    r2f = np.empty(len(d['y']), dtype=object)
    for f, rr in d['fr'].items():
        r2f[rr] = f
    c, u = pd.factorize(r2f); return c, len(u)

def romano_wolf(hyp, B=RW_B, seed=SEED):
    keys = list(hyp); S = len(keys)
    t0 = np.array([cluster_ols_fast(hyp[k]['y'], hyp[k]['X'], *_codes(hyp[k]))[1] for k in keys])
    firms = np.unique(np.concatenate([hyp[k]['firms'] for k in keys]))
    rng = np.random.default_rng(seed); bt = np.full((B, S), np.nan)
    for b in range(B):
        samp = rng.choice(firms, size=len(firms), replace=True)
        for si, k in enumerate(keys):
            d = hyp[k]; pres = [f for f in samp if f in d['fr']]
            if not pres: continue
            idx = np.concatenate([d['fr'][f] for f in pres])
            seq = np.concatenate([[i] * len(d['fr'][f]) for i, f in enumerate(pres)])
            bt[b, si] = cluster_ols_fast(d['y'][idx], d['X'][idx], seq, len(pres))[1]
    cent = bt - t0[None, :]; order = np.argsort(-np.abs(t0)); padj = np.empty(S); active = list(order)
    for kk in range(S):
        sk = order[kk]; m = np.nanmax(np.abs(cent[:, active]), axis=1)
        padj[sk] = np.nanmean(m >= np.abs(t0[sk])); active.remove(sk)
    run = 0.0
    for kk in range(S):
        sk = order[kk]; run = max(run, padj[sk]); padj[sk] = run
    return pd.Series(t0, index=keys), pd.Series(padj, index=keys)

print('Estimator + RW loaded.')

Estimator + RW loaded.


## 3 — Hypothesis-generating sample (FY23–FY24)

Restating the risk signal on the same window `09` used, at the 126-td horizon. This is
**not** evidence — it is where the hypothesis comes from. Each pair shown twice: the raw
cross-sectional coefficient, and the conditional one (with the pre-filing lagged-vol
control). Negative = better governance → lower risk.

In [5]:
def family_table(df, title):
    print(title); print('=' * 92)
    print(f'{"Index":8s} {"Risk measure":14s} {"raw beta":>10s} {"t":>7s} {"p":>7s}   '
          f'{"|+lagvol beta":>13s} {"t":>7s} {"p":>7s}   N')
    print('-' * 92)
    out = []
    for cg_col in FAMILY_IDX:
        for y in VOLS:
            b = reg(df, y, cg_col); l = reg(df, y, cg_col, LAGMAP[y])
            if b is None:
                print(f'{cg_col:8s} {y:14s}  n/a'); continue
            print(f'{cg_col:8s} {y:14s} {b["beta"]:>10.4f} {b["t"]:>7.2f} {b["p_raw"]:>7.3f}   '
                  f'{l["beta"]:>13.4f} {l["t"]:>7.2f} {l["p_raw"]:>7.3f}   {b["N"]}')
            out.append({'sample': title.split()[0], 'Index': cg_col, 'measure': y,
                        'beta': b['beta'], 't': b['t'], 'p_raw': b['p_raw'],
                        'beta_lag': l['beta'], 't_lag': l['t'], 'p_lag': l['p_raw'], 'N': b['N']})
    print('=' * 92)
    return pd.DataFrame(out)

train_tbl = family_table(panel_train, 'FY23-24 hypothesis-generating family')

FY23-24 hypothesis-generating family
Index    Risk measure     raw beta       t       p   |+lagvol beta       t       p   N
--------------------------------------------------------------------------------------------
TRINDEX  total_vol         -0.0007   -2.42   0.016         -0.0005   -2.16   0.032   395
TRINDEX  idio_vol          -0.0007   -2.48   0.014         -0.0005   -2.19   0.030   395
TRINDEX  downside_vol      -0.0002   -1.03   0.306         -0.0002   -0.88   0.379   395
TRINDEX  max_drawdown      -0.0125   -3.95   0.000         -0.0110   -3.54   0.000   395


OINDEX   total_vol         -0.0008   -2.44   0.016         -0.0006   -2.18   0.030   395
OINDEX   idio_vol          -0.0009   -2.71   0.007         -0.0007   -2.43   0.016   395
OINDEX   downside_vol      -0.0004   -2.03   0.043         -0.0004   -1.97   0.050   395
OINDEX   max_drawdown      -0.0021   -0.63   0.531         -0.0007   -0.20   0.839   395


## 4 — Held-out test (FY25)

FY25 was never used to form the hypothesis, so this is out-of-sample in the *temporal*
sense. Read it with the caveat that §6 quantifies: it is the **same firms** one year later,
not a fresh draw of firms — so a stable firm-level correlation will reproduce here partly by
construction. Treat this section as necessary-but-not-sufficient, and §6 as the real test.

In [6]:
test_tbl = family_table(panel_test, 'FY25 held-out confirmatory family')

# Confirmation scorecard: same sign as FY23-24 AND raw p<.05 in FY25
merged = train_tbl.merge(test_tbl, on=['Index', 'measure'], suffixes=('_tr', '_te'))
merged['same_sign'] = np.sign(merged['beta_tr']) == np.sign(merged['beta_te'])
merged['confirms_raw'] = merged['same_sign'] & (merged['p_raw_te'] < 0.05)
merged['confirms_lagged'] = merged['same_sign'] & (merged['p_lag_te'] < 0.05)
print('\nConfirmation scorecard (held-out FY25 vs FY23-24 hypothesis):')
print(f"  same negative sign in both samples : {merged['same_sign'].sum()}/8")
print(f"  confirm at raw p<.05 (no lag ctrl)  : {merged['confirms_raw'].sum()}/8  "
      f"-> {', '.join((merged.loc[merged['confirms_raw'],'Index']+' '+merged.loc[merged['confirms_raw'],'measure']).tolist()) or '(none)'}")
print(f"  confirm at raw p<.05 (+lag control) : {merged['confirms_lagged'].sum()}/8  "
      f"-> {', '.join((merged.loc[merged['confirms_lagged'],'Index']+' '+merged.loc[merged['confirms_lagged'],'measure']).tolist()) or '(none)'}")

FY25 held-out confirmatory family
Index    Risk measure     raw beta       t       p   |+lagvol beta       t       p   N
--------------------------------------------------------------------------------------------
TRINDEX  total_vol         -0.0009   -2.35   0.020         -0.0005   -1.50   0.136   199
TRINDEX  idio_vol          -0.0007   -1.89   0.060         -0.0005   -1.36   0.177   199
TRINDEX  downside_vol      -0.0003   -1.39   0.165         -0.0001   -0.69   0.488   199
TRINDEX  max_drawdown      -0.0173   -3.05   0.003         -0.0128   -2.19   0.030   199
OINDEX   total_vol         -0.0009   -2.24   0.026         -0.0007   -2.08   0.039   199
OINDEX   idio_vol          -0.0009   -2.05   0.041         -0.0007   -1.96   0.052   199
OINDEX   downside_vol      -0.0005   -1.50   0.135         -0.0004   -1.34   0.182   199


OINDEX   max_drawdown      -0.0072   -1.05   0.295         -0.0047   -0.66   0.513   199

Confirmation scorecard (held-out FY25 vs FY23-24 hypothesis):
  same negative sign in both samples : 8/8
  confirm at raw p<.05 (no lag ctrl)  : 4/8  -> TRINDEX total_vol, TRINDEX max_drawdown, OINDEX total_vol, OINDEX idio_vol
  confirm at raw p<.05 (+lag control) : 2/8  -> TRINDEX max_drawdown, OINDEX total_vol


## 5 — Familywise correction on the held-out family

Eight pre-declared tests, so raw p-values overstate significance. Romano-Wolf (firm
bootstrap, B=2000) on the FY25 family, for both the no-control and lagged-control specs.
This is the number that supports any actual claim. (Run here, unlike `16`, because the
result is *not* null — a correction is exactly what's needed to see how much survives.)

In [7]:
pairs = [(cg_col, y) for cg_col in FAMILY_IDX for y in VOLS]
rw_rows = []
for lagged, label in [(False, 'no lag control'), (True, '+ lagged vol')]:
    t0 = time.time()
    hyp = build_hyp(panel_test, pairs, lagged=lagged)
    tt, pp = romano_wolf(hyp)
    r = pd.DataFrame({'t': tt, 'p_rw': pp}).reset_index()
    r[['Index', 'measure']] = pd.DataFrame(r['index'].tolist(), index=r.index)
    r = r.drop(columns='index').sort_values('p_rw'); r['spec'] = label
    rw_rows.append(r)
    print(f'FY25 held-out RW [{label}] ({time.time()-t0:.0f}s, B={RW_B}): '
          f'survivors p<.10 = {(r["p_rw"]<.10).sum()}/8, p<.05 = {(r["p_rw"]<.05).sum()}/8, min p_rw = {r["p_rw"].min():.4f}')
    print(r[['Index', 'measure', 't', 'p_rw']].round(4).to_string(index=False))
    print()
rw_test = pd.concat(rw_rows, ignore_index=True)

FY25 held-out RW [no lag control] (14s, B=2000): survivors p<.10 = 1/8, p<.05 = 1/8, min p_rw = 0.0335
  Index      measure       t   p_rw
TRINDEX max_drawdown -3.0483 0.0335
TRINDEX    total_vol -2.3534 0.1285
 OINDEX    total_vol -2.2374 0.1480
 OINDEX     idio_vol -2.0546 0.2025
TRINDEX     idio_vol -1.8941 0.2315
TRINDEX downside_vol -1.3938 0.3690
 OINDEX downside_vol -1.5025 0.3690
 OINDEX max_drawdown -1.0504 0.3690



FY25 held-out RW [+ lagged vol] (16s, B=2000): survivors p<.10 = 0/8, p<.05 = 0/8, min p_rw = 0.1945
  Index      measure       t   p_rw
TRINDEX max_drawdown -2.1909 0.1945
 OINDEX    total_vol -2.0787 0.2005
 OINDEX     idio_vol -1.9582 0.2465
TRINDEX    total_vol -1.4973 0.4470
TRINDEX     idio_vol -1.3553 0.5180
 OINDEX downside_vol -1.3384 0.5180
TRINDEX downside_vol -0.6943 0.7455
 OINDEX max_drawdown -0.6558 0.7455



## 6 — The threat Romano-Wolf cannot see: persistence, and the within-firm test

RW corrects for **multiple testing**. It says nothing about a separate and, here, more
serious problem: the held-out FY25 sample is **the same 210 firms**, and both governance
scores and volatility are highly persistent at the firm level. If a firm is intrinsically
low-risk *and* scores well on governance for some third reason, that association reproduces
in FY25 mechanically — no fresh information required. §6.1 measures how persistent these
variables actually are; §6.2 runs the test that removes the problem entirely.

**The within-firm (first-difference) test** asks the question the cross-section cannot:
*when a firm's own governance score changes, does its own risk change?* Differencing sweeps
out every time-invariant firm characteristic — industry, business model, and any unobserved
trait driving both. If the cross-sectional association reflects a real governance→risk
mechanism, it should survive here. If it only reflects stable firm heterogeneity, it won't.

In [8]:
# 6.1 — how persistent are these variables at the firm level?
print('Firm-level year-over-year correlation (persistence):')
print('-' * 58)
for col in ['TRINDEX', 'OINDEX'] + VOLS:
    piv = panel.pivot_table(index='BSE Code', columns='FY', values=col)
    c1 = piv[['FY23', 'FY24']].dropna().corr().iloc[0, 1]
    c2 = piv[['FY24', 'FY25']].dropna().corr().iloc[0, 1]
    note = '   <-- effectively a FIXED firm attribute' if min(c1, c2) > 0.98 else ''
    print(f'  {col:14s} corr(FY23,FY24)={c1:+.2f}   corr(FY24,FY25)={c2:+.2f}{note}')
print('-' * 58)
print('The higher these are, the less independent information the FY25 "held-out" test')
print('actually carries — a persistent x persistent association re-appears by construction.')

Firm-level year-over-year correlation (persistence):
----------------------------------------------------------
  TRINDEX        corr(FY23,FY24)=+0.93   corr(FY24,FY25)=+0.94
  OINDEX         corr(FY23,FY24)=+1.00   corr(FY24,FY25)=+1.00   <-- effectively a FIXED firm attribute
  total_vol      corr(FY23,FY24)=+0.66   corr(FY24,FY25)=+0.64
  idio_vol       corr(FY23,FY24)=+0.60   corr(FY24,FY25)=+0.57
  downside_vol   corr(FY23,FY24)=+0.27   corr(FY24,FY25)=+0.39
  max_drawdown   corr(FY23,FY24)=+0.26   corr(FY24,FY25)=+0.23
----------------------------------------------------------
The higher these are, the less independent information the FY25 "held-out" test
actually carries — a persistent x persistent association re-appears by construction.


In [9]:
# 6.2 — within-firm first-difference test (pooled FY23->FY24 and FY24->FY25)
NUM = VOLS + FAMILY_IDX
piv = panel.pivot_table(index='BSE Code', columns='FY', values=NUM)
print('WITHIN-FIRM FIRST-DIFFERENCE: does a CHANGE in governance predict a CHANGE in risk?')
print('=' * 84)
fd_rows = []
for cg_col in FAMILY_IDX:
    within_sd = np.nanmean([(piv[(cg_col, b)] - piv[(cg_col, a)]).std()
                            for a, b in [('FY23', 'FY24'), ('FY24', 'FY25')]])
    for y in VOLS:
        parts = []
        for a, b in [('FY23', 'FY24'), ('FY24', 'FY25')]:
            parts.append(pd.concat([(piv[(cg_col, b)] - piv[(cg_col, a)]).rename('dcg'),
                                    (piv[(y, b)] - piv[(y, a)]).rename('dy')], axis=1).dropna())
        s = pd.concat(parts)
        r = sm.OLS(s['dy'], sm.add_constant(s['dcg'])).fit(cov_type='HC1')
        beta, tval, pval = r.params['dcg'], r.tvalues['dcg'], r.pvalues['dcg']
        flag = ' <-- p<.05' if pval < .05 else ''
        warn = '  [UNIDENTIFIED: regressor has ~no within-firm variation]' if within_sd < 0.05 else ''
        print(f'  d{cg_col:8s} -> d_{y:14s} beta={beta:+.5f} t={tval:+.2f} p={pval:.3f} N={len(s)}{flag}{warn}')
        fd_rows.append({'Index': cg_col, 'measure': y, 'beta_fd': beta, 't_fd': tval,
                        'p_fd': pval, 'N': len(s), 'within_firm_sd': within_sd})
fd_tbl = pd.DataFrame(fd_rows)
print('=' * 84)
print(f"Within-firm SD of the differenced score: "
      + ', '.join(f"{c}={fd_tbl[fd_tbl['Index']==c]['within_firm_sd'].iloc[0]:.3f}" for c in FAMILY_IDX))
print('A near-zero within-firm SD means the index is a fixed firm attribute in this data:')
print('the first-difference coefficient is then dividing by noise and is NOT interpretable')
print('(expect implausible magnitudes and unstable signs) — the effect is simply not')
print('identifiable within-firm, which is itself the finding.')
fd_tbl.to_csv(PROC / 'volatility_within_firm_firstdiff.csv', index=False)
print('\nSaved -> volatility_within_firm_firstdiff.csv')

WITHIN-FIRM FIRST-DIFFERENCE: does a CHANGE in governance predict a CHANGE in risk?
  dTRINDEX  -> d_total_vol      beta=-0.00085 t=-1.05 p=0.295 N=448
  dTRINDEX  -> d_idio_vol       beta=-0.00094 t=-1.19 p=0.235 N=448
  dTRINDEX  -> d_downside_vol   beta=-0.00051 t=-0.80 p=0.422 N=448
  dTRINDEX  -> d_max_drawdown   beta=-0.01725 t=-0.90 p=0.368 N=448
  dOINDEX   -> d_total_vol      beta=+0.30019 t=+6.93 p=0.000 N=448 <-- p<.05  [UNIDENTIFIED: regressor has ~no within-firm variation]
  dOINDEX   -> d_idio_vol       beta=+0.20513 t=+4.70 p=0.000 N=448 <-- p<.05  [UNIDENTIFIED: regressor has ~no within-firm variation]
  dOINDEX   -> d_downside_vol   beta=+0.10752 t=+3.10 p=0.002 N=448 <-- p<.05  [UNIDENTIFIED: regressor has ~no within-firm variation]
  dOINDEX   -> d_max_drawdown   beta=+6.51958 t=+7.43 p=0.000 N=448 <-- p<.05  [UNIDENTIFIED: regressor has ~no within-firm variation]
Within-firm SD of the differenced score: TRINDEX=0.367, OINDEX=0.005
A near-zero within-firm SD means th

## 7 — Secondary check: the other three sub-indices (exploratory)

For completeness, the same held-out FY25 test on AINDEX/BINDEX/CINDEX — **not** part of the
pre-declared family, reported exploratory-only, so a stray low p here is not a claim.

In [10]:
print(f'{"Index":8s} {"Risk measure":14s} {"beta":>10s} {"t":>7s} {"p_raw":>7s}   N   (FY25, exploratory)')
print('-' * 68)
for cg_col in ['AINDEX', 'BINDEX', 'CINDEX']:
    for y in VOLS:
        b = reg(panel_test, y, cg_col)
        if b:
            flag = ' *' if b['p_raw'] < 0.05 else ''
            print(f'{cg_col:8s} {y:14s} {b["beta"]:>10.4f} {b["t"]:>7.2f} {b["p_raw"]:>7.3f}   {b["N"]}{flag}')
print('(* raw p<.05, uncorrected, exploratory — not part of the confirmatory family)')

Index    Risk measure         beta       t   p_raw   N   (FY25, exploratory)
--------------------------------------------------------------------
AINDEX   total_vol         -0.0005   -1.53   0.129   199
AINDEX   idio_vol          -0.0006   -1.76   0.080   199
AINDEX   downside_vol      -0.0002   -0.79   0.433   199
AINDEX   max_drawdown      -0.0071   -1.27   0.206   199
BINDEX   total_vol         -0.0006   -1.37   0.172   199
BINDEX   idio_vol          -0.0005   -1.16   0.248   199
BINDEX   downside_vol      -0.0004   -1.20   0.232   199
BINDEX   max_drawdown      -0.0067   -0.92   0.359   199
CINDEX   total_vol         -0.0002   -0.45   0.653   207
CINDEX   idio_vol          -0.0001   -0.35   0.725   207
CINDEX   downside_vol       0.0000    0.01   0.990   207
CINDEX   max_drawdown      -0.0027   -0.41   0.682   207
(* raw p<.05, uncorrected, exploratory — not part of the confirmatory family)


In [11]:
# Persist
vol.to_csv(PROC / 'volatility_outcomes_by_fy.csv', index=False)
train_tbl.to_csv(PROC / 'volatility_hypothesis_fy2324.csv', index=False)
test_tbl.to_csv(PROC / 'volatility_heldout_fy25.csv', index=False)
rw_test.to_csv(PROC / 'volatility_heldout_romano_wolf.csv', index=False)
print('Saved -> volatility_outcomes_by_fy.csv, volatility_hypothesis_fy2324.csv,')
print('         volatility_heldout_fy25.csv, volatility_heldout_romano_wolf.csv')

Saved -> volatility_outcomes_by_fy.csv, volatility_hypothesis_fy2324.csv,
         volatility_heldout_fy25.csv, volatility_heldout_romano_wolf.csv


## 8 — Verdict

Stated at the strength the evidence actually supports, not higher:

- The main study's **return/alpha null stands** — governance did not predict returns.
- **What survives:** a **cross-sectional association** — better-governed firms (TRINDEX,
  OINDEX) *are* lower-risk firms. The negative sign holds on all 8 pre-declared pairs in
  FY25, and TRINDEX → max-drawdown clears familywise correction out-of-sample (§5). As a
  descriptive statement about which firms carry less risk, this is solid.
- **What does not survive:** any *temporal or causal* reading. §6 shows the governance
  indices are near-frozen at the firm level (OINDEX ~1.00 year-over-year), so the FY25
  "replication" carries much less independent information than the raw p-values imply, and
  the within-firm first-difference test — the one that removes stable firm heterogeneity —
  finds **no effect for TRINDEX** and **cannot identify OINDEX at all** for want of
  within-firm variation. Adding the lagged-volatility control to the held-out cross-section
  points the same way: 0/8 survive correction (§5).
- **Therefore:** *"well-governed firms tend to be lower-risk firms"* is supported.
  *"Improving governance reduces risk"* is **not** — this design cannot distinguish a
  governance effect from any stable firm trait correlated with both, and the honest reading
  is that much of the association is such heterogeneity.
- **Why this still matters:** it is a sharper, better-identified negative than the returns
  null. Governance behaves like a *risk attribute* of a firm, not a lever that moves risk —
  and the data say the attribute is nearly time-invariant, which is itself informative about
  what these indices measure.
- **What would settle it:** firms with genuine governance *transitions* (a real shock —
  board overhaul, promoter-pledge unwind, auditor resignation), or a longer panel where
  within-firm variation accumulates. Both are outside this dataset's T=3.
- **Caveats unchanged:** survivorship-conditioned large-cap universe (README Limitations §1)
  attenuates effects; the 252-td horizon is the natural next robustness cut as more FY25
  history elapses.